# 06 — Reject Inference

In [ ]:
# Cell 1 — setup
import sys, pathlib
sys.path.append(str(pathlib.Path.cwd().parent / "src"))
from scorelab import config, data
con = data.connect()            # read-only DuckDB connection to data/lc.duckdb

In [ ]:
# Cell 2 — sanity check
print(con.sql("SHOW TABLES"))
print(con.sql("SELECT COUNT(*) AS n, MIN(issue_d), MAX(issue_d) FROM accepted"))

In [ ]:
# Cell 3 — common imports and settings for this notebook
import json, warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.width", 180); pd.set_option("display.max_columns", 40); pd.set_option("display.max_rows", 200)
from scorelab import target, attributes as A, binning as B, scorecard as S, validation as V, fairness as F
from scorelab import challengers as C, reject_inference as R, monitoring as M, report as REP
MODELS = config.MODELS_DIR; MODELS.mkdir(exist_ok=True)
SAMPLE_FRAC = None     # None = full data (recommended for the fitting notebooks); e.g. 0.10 for a quick pass
cols = [r[0] for r in con.execute("DESCRIBE v_target").fetchall()]
assert "label" in cols, "v_target is the old stub: close this kernel and run `python scripts/refresh_views.py`"
def load_frame(splits=("train", "validate", "oot"), **kw):
    """Observable, non-excluded loans with attributes built; indeterminates dropped unless asked."""
    f = target.modeling_frame(con, splits=splits, sample_frac=SAMPLE_FRAC, **kw)
    return A.build_attributes(f)
MEANINGS = {a.name: a.meaning for a in A.REGISTRY}
DIRECTIONS = {a.name: a.direction for a in A.REGISTRY}

## What this notebook settles

Whether and how reject inference changes the card, and what population each number in the pilot report describes. The real rejected file has no bureau attributes, so the methods are validated on a **simulated reject region with hidden truth**: the riskiest share of the accepted population by FICO is treated as declined, the known-good-bad (KGB) card is built on the rest, each inference method fills the declines, and every variant is compared on the full population where the truth is known.

Decisions: **L — reject rate and incumbent for the simulation**, **M — which method, if any, is defensible for the pilot**.

In [ ]:
frame = load_frame()
tr, va, oot = (frame[frame.split == s] for s in ("train", "validate", "oot"))
card, specs = S.load_card(MODELS / "champion"); attrs = card.attributes
def fit_fn(f):
    Wf = B.transform_woe(f, {a: specs[a] for a in attrs})
    return S.fit_logistic(Wf, f["bad"], attrs, sample_weight=f["weight"] if "weight" in f else None)
def score_fn(c, f): return S.score(f, c, specs)["p_bad"].to_numpy()

In [ ]:
# Evidence — how far the booked population is from through-the-door, from the real rejected file
ttd = R.through_the_door_profile(con, years=tuple(config.TRAIN_VINTAGES) + (config.VALIDATE_VINTAGE, config.OOT_VINTAGE))
display(ttd)
fig, ax = plt.subplots(figsize=(9, 4)); ax.plot(ttd.fico_band, ttd.share_rejected, label="rejected"); ax.plot(ttd.fico_band, ttd.share_accepted, label="accepted")
ax.set_xlim(500, 850); ax.set_xlabel("FICO band"); ax.set_ylabel("share"); ax.legend(); plt.show()

### Decision L — Simulation design

Reject rate (Lending Club's real acceptance rate is far lower than any lender pilot; the simulation should mimic a lender's cutoff, 10–30%), and the incumbent used to decline (FICO midpoint, as in the pilot).

**Your reasoning:**

_(write here)_

In [ ]:
REJECT_RATE = 0.20
dev = pd.concat([tr, va])
sim = R.simulate_reject_region(dev, incumbent="fico_mid", reject_rate=REJECT_RATE)
print(f"declined {sim.declined.mean():.1%} · true bad rate declined {sim.loc[sim.declined, 'bad_hidden'].mean():.2%} · approved {sim.loc[~sim.declined, 'bad_hidden'].mean():.2%}")
cmp = R.compare_methods(sim, fit_fn, score_fn, oot, multipliers=(1.0, 1.5, 2.0))
display(cmp); cmp.to_csv(config.DOCS_DIR / "exhibits" / "reject_inference_comparison.csv", index=False)

In [ ]:
# Evidence — do decisions change? KGB card vs fuzzy-augmented card at the pilot approval rate
kgb = fit_fn(sim[~sim.declined].assign(weight=1.0))
p_dec = score_fn(kgb, sim[sim.declined])
aug = fit_fn(R.fuzzy_augmentation(sim, p_dec))
display(R.swap_vs_kgb(sim, score_fn(kgb, sim), score_fn(aug, sim), approval_rate=config.APPROVAL_RATE_FOR_SWAP))

### Decision M — What the pilot reports

Given the comparison: does any method recover rank order on the declines that the KGB card does not? Is the inferred bad rate on declines close to the hidden truth? State which population (booked only / augmented) the pilot numbers describe, and whether to show both.

**Your reasoning:**

_(write here)_